# 1.0 — Exploratory Data Analysis

**RetrofitTrust Birmingham PoC** · MSc AI dissertation (BCU)

This notebook is for **narrative exploration only**. All reusable logic lives in `src/retrofittrust/` and is executed via `scripts/01–03`.

## Purpose

Document initial inspection of the three source datasets before and after the Birmingham-filtered merge on **2021 LSOA code**:

1. **EPC Domestic** — property-level certificates (filtered to Birmingham LA)
2. **English Indices of Deprivation 2025** — LSOA-level IMD score, rank, decile, income domain
3. **ONS Census 2021 (Nomis)** — TS054 Tenure and Central Heating at LSOA level

## Known limitations to record here

- **Ecological fallacy:** LSOA-level IMD does not describe individual households.
- **EPC coverage bias:** certificates exist only where triggered (sale, let, new build).
- **Performance gap:** modelled EPC energy use diverges from metered consumption (~16% gas, ~31% electric).

## Expected outputs from pipeline scripts

| Script | Checkpoint | Output |
|--------|------------|--------|
| `01_ingest_and_merge.py` | §8.1 | `data/processed/merged_lsoa.parquet` |
| `02_train_quality_screen.py` | §8.2 | `data/processed/quality_flagged.parquet` |
| `03_train_ranking_model.py` | §8.3 | `models/lgbm_ranker.joblib` |

Run the scripts first; use this notebook to summarise row counts, missingness patterns, and distributional plots for the dissertation Methods chapter.

## Suggested EDA sections (fill after pipeline run)

1. **Row counts and join retention** — compare EPC Birmingham rows before/after LSOA join with IMD and Census.
2. **Missingness heatmap** — note fields with high missing rates; these feed missingness-indicator columns in preprocessing.
3. **EPC rating distribution** — current vs potential energy rating; gap as retrofit-need proxy.
4. **IMD income domain vs EPC efficiency** — area-level scatter; discuss ecological fallacy explicitly.
5. **Quality-screen flagged rate** — compare consensus flag rate to literature range (~27–60%).

> Do not duplicate training or anomaly-detection logic here — import from `retrofittrust` modules if ad-hoc plots are needed.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from retrofittrust.config import DATA_PROCESSED, DATA_RAW, SEED  # noqa: E402

AUDIT_PATH = DATA_PROCESSED / "join_audit.json"
MERGED_PATH = DATA_PROCESSED / "merged_lsoa.parquet"

print(f"SEED={SEED} | project root={PROJECT_ROOT}")

## 1. Join audit (checkpoint 1)

Run `python scripts/01_ingest_and_merge.py` first. The pipeline writes `data/processed/join_audit.json` with per-step row counts — cite this file in the Methods chapter.

**Design choices logged here:**
- Left joins from EPC (property-level); unmatched rows are **kept** with nulls and `*_matched` flags — no silent drops.
- IMD and Census are **LSOA-level** (659 Birmingham LSOAs in this extract); attaching them to dwellings is an ecological join.
- **1,153 EPC rows** lack `lsoa21cd` in the source parquet; the ONS postcode lookup still has null LSOA for those postcodes, so they remain unmatched (1,157 including LSOAs outside the IMD extract).
- **Geography GeoJSON is not present** — choropleth mapping needs `data/external/lsoa_birmingham.geojson` (ONS 2021 LSOA BGC).

In [ ]:
if not AUDIT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {AUDIT_PATH}. Run scripts/01_ingest_and_merge.py from the project root."
    )

with AUDIT_PATH.open(encoding="utf-8") as handle:
    audit = json.load(handle)

print("Join key:", audit["join_key"])
print("\nSource row counts:")
for name, stats in audit["sources"].items():
    print(f"  {name:10s}", stats)

print("\nJoin steps (no row loss on left join):")
for step in audit["join_steps"]:
    print(
        f"  {step['join']:28s} matched={step['matched']:,} "
        f"unmatched={step['unmatched_left']:,} rows={step['rows_after']:,}"
    )

print("\nCoverage (LSOAs with area data but no EPC are logged, not deleted):")
print(json.dumps(audit.get("coverage", {}), indent=2))

print("\nCaveats recorded in audit:")
for line in audit.get("caveats", []):
    print(" •", line)

## 2. Merged dataset — Birmingham LSOA counts & nulls

In [ ]:
df = pd.read_parquet(MERGED_PATH)

summary = {
    "property_rows": len(df),
    "unique_lsoa21cd": int(df["lsoa21cd"].nunique()),
    "birmingham_imd_lsoas": audit["sources"]["imd"]["unique_lsoa"],
    "imd_matched_rows": int(df["imd_matched"].sum()) if "imd_matched" in df.columns else None,
    "census_matched_rows": int(df["census_matched"].sum()) if "census_matched" in df.columns else None,
    "non_null_priority_scores": int(df["retrofit_priority_score"].notna().sum()),
}
pd.Series(summary, name="merged_lsoa_summary")

In [ ]:
nulls = pd.DataFrame(audit.get("nulls_key_columns", {})).T
nulls

## 3. Distributional checks (dissertation figures)

These plots support the Methods chapter. They are **not** used for modelling here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

if "current_energy_rating" in df.columns:
    df["current_energy_rating"].astype(str).str.upper().value_counts().sort_index().plot.bar(
        ax=axes[0], title="EPC current rating (property-level)"
    )

if "imd_income_score" in df.columns:
    df.groupby("lsoa21cd")["imd_income_score"].first().dropna().hist(
        bins=30, ax=axes[1], edgecolor="white"
    )
    axes[1].set_title("IMD income score (LSOA-level — ecological proxy)")

plt.tight_layout()
plt.show()